# 🔮 Finance-Pro: Meta Prophet Time-Series Trend & Seasonality Pipeline

Notebook ini menyediakan pipeline end-to-end terstruktur untuk analisis tren & peramalan (forecasting) saham IDX menggunakan **Meta Prophet**:
1. **Environment Setup**: Petunjuk & verifikasi modul Meta `prophet`.
2. **Load Data**: Mengambil data OHLCV dari SQLite database (`data/ihsg_trading.db`).
3. **Data Cleaning & Reshaping**: Format khusus Prophet (`ds` untuk tanggal, `y` untuk harga close).
4. **Prophet Model Training**: Pemodelan Tren Additive/Multiplicative, Daily/Weekly/Yearly Seasonality, & Libur IDX.
5. **Forecast & Decomposition**: Prediksi 30 hari ke depan (Future DataFrame) + Komponen Tren & Musiman.
6. **Evaluation Metrics**: Menghitung RMSE, MAE, dan MAPE dari peramalan.

## Cell 1: Environment Setup & Package Installation Instructions

> **Instruksi**: Jika `prophet` belum terpasang di environment Anda, jalankan perintah pip berikut di terminal:
> ```bash
> pip install prophet pandas numpy matplotlib
> ```

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Ensure project root is in path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pipeline.storage import StorageManager
from pipeline.data_cleaner import DataCleaner

from model.registry import ModelRegistry
print("✓ Environment & base modules successfully loaded!")

## Cell 2: Load Data dari SQLite Database

In [ ]:
storage = StorageManager()
available_tickers = storage.get_available_tickers()
print(f"Database Path  : {storage.db_path}")
print(f"Total Tickers  : {len(available_tickers)} emitens")

raw_close_prices = storage.load_close_prices()
print(f"Raw Data Shape : {raw_close_prices.shape}")
raw_close_prices.head()

## Cell 3: Data Cleaning & Format Conversion (ds, y)

In [ ]:
cleaner = DataCleaner(min_price=200.0)
cleaned_prices = cleaner.clean(raw_close_prices)

# Select a specific benchmark emiten (e.g. BBCA.JK)
ticker = "BBCA.JK"
df_single = cleaned_prices[[ticker]].dropna().reset_index()
df_single.columns = ["ds", "y"]
df_single["ds"] = pd.to_datetime(df_single["ds"])

print(f"Prophet DataFrame Ready for {ticker}:")
print(df_single.head())

## Cell 4: Meta Prophet Model Training

In [ ]:
try:
    from prophet import Prophet
    
    model = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=True,
        changepoint_prior_scale=0.05
    )
    model.fit(df_single)
    print(f"✓ Prophet Model successfully trained on {ticker} ({len(df_single)} trading days)!")
except ImportError:
    print("⚠ Prophet belum terinstall. Silakan install via `pip install prophet`.")

## Cell 5: Forecasting 30 Hari Ke Depan & Component Decomposition

In [ ]:
try:
    future = model.make_future_dataframe(periods=30)
    forecast = model.predict(future)
    
    print("Forecast Results (30-day ahead horizon):")
    display(forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(10))
    
    fig1 = model.plot(forecast)
    plt.title(f"{ticker} 30-Day Trend & Price Forecast", fontsize=14)
    plt.show()
    
    fig2 = model.plot_components(forecast)
    plt.show()
except NameError:
    print("Model belum dilatih karena pustaka prophet belum terinstall.")

## Cell 6: Forecast Accuracy Metrics (RMSE, MAE, MAPE)

In [ ]:
try:
    y_true = df_single["y"].values
    y_pred = forecast.iloc[:len(y_true)]["yhat"].values
    
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    
    print("=================== PROPHET FORECAST METRICS ===================")
    print(f"Mean Absolute Error (MAE)       : Rp {mae:.2f}")
    print(f"Root Mean Squared Error (RMSE)  : Rp {rmse:.2f}")
    print(f"Mean Absolute Percentage (MAPE) : {mape:.2f}%")
except NameError:
    pass

## Cell 7: Save & Register Prophet Model to Model Registry

Model Prophet yang telah dilatih akan disimpan sebagai artifact `.pkl` dan didaftarkan ke `ModelRegistry` beserta metrik akurasi.

In [ ]:
import pickle

output_dir = os.path.join(PROJECT_ROOT, "artifacts", "saved_models")
os.makedirs(output_dir, exist_ok=True)

try:
    save_path = os.path.join(output_dir, f"prophet_{ticker.replace('.', '_')}_notebook.pkl")
    with open(save_path, "wb") as f:
        pickle.dump({"model": model, "ticker": ticker}, f)
    print(f"✓ Prophet artifact saved: {save_path}")

    registry = ModelRegistry(
        artifacts_dir=output_dir,
        db_path=os.path.join(PROJECT_ROOT, "artifacts", "registry.db"),
    )
    mv = registry.register(
        model_type="prophet",
        artifact_path=save_path,
        metrics={"mae": float(mae), "rmse": float(rmse), "mape": float(mape)},
        description=f"Prophet trend model for {ticker} from Notebook 03",
        tags=[ticker],
    )
    print(f"📦 Registered as: {mv.version_id} (stage={mv.stage})")
    print(f"   MAE: Rp {mae:.2f} | RMSE: Rp {rmse:.2f} | MAPE: {mape:.2f}%")
except Exception as e:
    print(f"⚠ Save/register skipped: {e}")